# mine-tuning-model (RTX 5060 Ti 최적화 버전 — Qwen3-9B)

## 사전 준비 (터미널에서 먼저 실행)

```bash
# 1. 가상환경 생성 및 활성화
python -m venv venv
source venv/Scripts/activate

# 2. PyTorch 설치 — RTX 5060 Ti (Blackwell)는 CUDA 12.8 필요!
#    nvidia-smi 실행해서 CUDA 버전 확인 후 아래 명령어 선택

# ✅ CUDA 12.8 (RTX 5060 Ti 권장)
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

# 3. Jupyter 설치
pip install jupyter

# vs code 커널 등록
python -m ipykernel install --user --name venv --display-name "mine-tuning (venv)"

# 4. 노트북 실행
jupyter notebook
```

## 0. GPU 확인 (제일 먼저 실행!)

In [ ]:
import subprocess
import sys

# GPU 확인
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(result.stdout)
except FileNotFoundError:
    print('⚠️  nvidia-smi를 찾을 수 없습니다. NVIDIA 드라이버가 설치되어 있는지 확인하세요.')

# PyTorch CUDA 확인
import torch
print(f'PyTorch 버전: {torch.__version__}')
print(f'CUDA 버전:    {torch.version.cuda}')
print(f'CUDA 사용 가능: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU 이름: {gpu_name}')
    print(f'GPU VRAM: {vram_gb:.1f}GB')

    # RTX 5060 Ti는 bf16 지원 확인
    bf16_support = torch.cuda.is_bf16_supported()
    print(f'bf16 지원: {bf16_support}')  # True여야 정상
else:
    print('❌ CUDA를 사용할 수 없습니다.')
    print('   → 사전 준비 단계에서 CUDA 12.8 버전 PyTorch를 설치했는지 확인하세요.')
    sys.exit('GPU가 필요합니다.')

## 1. 필수 라이브러리 설치

> 💡 처음 한 번만 실행하면 돼요!

In [ ]:
!pip install -U transformers -q
!pip install trl==0.17.0 datasets peft accelerate bitsandbytes -q

In [ ]:
!pip freeze > requirements-ft-9B.txt

## 2. 데이터 로드 및 Instruction 형식 변환

In [ ]:
from datasets import load_dataset

# 시스템 프롬프트 — 여기만 수정하면 전체 적용
SYSTEM_PROMPT = (
     "/no_think\n"
    "You are an expert Minecraft assistant with deep knowledge of all game mechanics, "
    "crafting recipes, biomes, mobs, and strategies.\n\n"
    "When answering:\n"
    "- Be specific and accurate about game mechanics\n"
    "- Include exact crafting recipes when relevant (materials and placement)\n"
    "- For survival tips, prioritize safety and efficiency\n"
    "- Keep answers clear and beginner-friendly unless the question is advanced"
)

def format_instruction(example):
    return {
        'text': (
            f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
            f"<|im_start|>user\n{example['question']}<|im_end|>\n"
            f"<|im_start|>assistant\n{example['answer']}<|im_end|>"
        )
    }

ds = load_dataset('ddorin/minecraft-question-answer-260k')
train_data = ds['train'].map(format_instruction, remove_columns=['question', 'answer', 'source'])

print(f'✅ 데이터 변환 완료: {len(train_data):,}개')
print('\n샘플 확인:')
print(train_data[0]['text'])


## 3. 모델, 토크나이저 로드

> 💡 4bit 양자화로 로드해서 VRAM을 절약해요. 9B 모델이 약 6~7GB로 줄어들어요.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import gc

# VRAM 비우기
gc.collect()
torch.cuda.empty_cache()

model_id = 'Qwen/Qwen3.5-9B'

# 4bit 양자화 설정
# RTX 5060 Ti (Blackwell)는 bf16 지원 → compute_dtype을 bf16으로 설정!
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,   # ✅ 5060 Ti는 bf16 사용 (fp16보다 안정적)
    bnb_4bit_use_double_quant=True,
)

print(f'모델 로딩 중: {model_id}')
print('처음 실행 시 약 19GB 다운로드가 필요해요...')

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={'': 0},  # GPU 0번에 전체 로드
)

model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={'use_reentrant': False}
)
model.enable_input_require_grads()

used  = torch.cuda.memory_allocated() / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'\n✅ 모델 로드 완료!')
print(f'VRAM 사용: {used:.1f}GB / {total:.1f}GB')

## 4. LoRA 어댑터 설정

> 💡 전체 모델을 학습하는 게 아니라 LoRA라는 작은 어댑터만 학습해요.
> 덕분에 VRAM을 훨씬 적게 쓰면서도 파인튜닝 효과를 얻을 수 있어요!

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 4bit 학습 준비
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        'q_proj', 'k_proj',
        'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj'
    ],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# 전체 파라미터 중 약 1~2%만 학습된다고 나오면 정상이에요!

## 5. 학습 실행 (SFTTrainer)

### ⚙️ 매 세션마다 아래 변수만 수정하세요

In [7]:
from trl import SFTTrainer, SFTConfig
import os

# ============================================================
# 저장 경로 — 실행마다 run_001, run_002 ... 자동 증가
# ============================================================
base = r"C:\mine-tuning-output\full_train"
n = 1
while os.path.exists(f"{base}_{n:03d}"):
    n += 1
OUTPUT_DIR = f"{base}_{n:03d}"
os.makedirs(OUTPUT_DIR)

print(f"📦 전체 학습 데이터: {len(train_data):,}건")
print(f"📁 저장 경로: {OUTPUT_DIR}")

# ============================================================
# 학습 설정
# ============================================================
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,                 # 전체 데이터 1회 학습 (수렴까지는 3 권장)
    per_device_train_batch_size=2,      # 9B 모델: VRAM 절약을 위해 2으로 조정
    gradient_accumulation_steps=8,      # 실질 배치 크기 = 16 (9B 조정)
    learning_rate=2e-4,
    bf16=True,                          # RTX 5060 Ti (Blackwell) 권장
    fp16=False,                         # bf16 사용 시 반드시 False
    logging_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    warmup_ratio=0.03,                  # 전체 스텝의 3% 워밍업
    lr_scheduler_type="cosine",
    report_to="none",
    dataloader_num_workers=0,           # Windows 멀티프로세싱 이슈 방지
    optim="paged_adamw_8bit",
)

# ============================================================
# Trainer 생성 및 학습
# ============================================================
trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    args=training_args,
    processing_class=tokenizer,
)

print("🚀 학습 시작!")
trainer.train()

# 최종 저장
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"\n✅ 학습 완료! → {OUTPUT_DIR}")


KeyboardInterrupt: 

## 6. 버전 확인

In [ ]:
import transformers, trl, torch

print(f'transformers: {transformers.__version__}')
print(f'trl:          {trl.__version__}')
print(f'torch:        {torch.__version__}')
print(f'CUDA:         {torch.version.cuda}')